
# Normalisierung der deutschen Titelannotationen

Dieses Notebook...
 - liest die drei vorhandenen Excel-Dateien ein,
 - verwendet bei SemEval das Worksheet `Sheet1`,
 - normalisiert `Headline_GER` technisch,
 - entfernt unsichtbare Unicode-Zeichen,
 - bereinigt Zeilenumbrüche, Tabs und mehrfachen Whitespace,
 - führt KEIN Lowercasing durch,
 - entfernt Duplikate nach den vereinbarten Regeln,
 - erstellt einen kurzen Qualitätsbericht,
 - speichert drei normalisierte Dateien sowie eine kombinierte Datei.

Die Daten werden nicht in Train/Dev/Test aufgeteilt.


Benötigte Pakete:
pandas
openpyxl



In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd



# ## 1. Konfiguration


In [2]:
from pathlib import Path

INPUT_DIR = Path.cwd()

INPUT_FILES = {
    "chn": INPUT_DIR / "chn_03_finale_titelannotationen.xlsx",
    "mfc": INPUT_DIR / "mfc_03_finale_titelannotationen.xlsx",
    "semeval": INPUT_DIR / "semeval_03_finale_titelannotationen_mit_3LLMs.xlsx",
}

# Ausgabeordner wird automatisch im selben Ordner erstellt.
OUTPUT_DIR = INPUT_DIR / "normalisierte_titelannotationen"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "chn": OUTPUT_DIR / "chn_03_finale_titelannotationen_normalisiert.xlsx",
    "mfc": OUTPUT_DIR / "mfc_03_finale_titelannotationen_normalisiert.xlsx",
    "semeval": OUTPUT_DIR / "semeval_03_finale_titelannotationen_normalisiert_mit_3LLMs.xlsx",
    "combined": OUTPUT_DIR / "titelannotationen_gesamt_normalisiert.xlsx",
}




TITLE_COLUMN = "Headline_GER"
ARTICLE_ID_COLUMN = "article_id"

LABEL_COLUMNS = [
    "Economic",
    "Capacity and resources",
    "Morality",
    "Fairness and equality",
    "Legality, constitutionality and jurisprudence",
    "Policy prescription and evaluation",
    "Crime and punishment",
    "Security and defense",
    "Health and safety",
    "Quality of life",
    "Cultural identity",
    "Public opinion",
    "Political",
    "External regulation and reputation",
]


# ============================================================
# 3. Excel-Sheets
# ============================================================

DATASET_CONFIG = {
    "chn": {
        "sheet": "Sheet1",
    },
    "mfc": {
        "sheet": "Sheet1",
    },
    "semeval": {
        "sheet": "Sheet1",
    },
}


# ============================================================
# 4. Eingabedateien prüfen
# ============================================================

print(f"Arbeitsverzeichnis: {INPUT_DIR}")
print()

for dataset_name, file_path in INPUT_FILES.items():
    if not file_path.exists():
        raise FileNotFoundError(
            f"{dataset_name.upper()}: Eingabedatei nicht gefunden:\n"
            f"{file_path}\n\n"
            "Bitte stelle sicher, dass Notebook und Excel-Dateien "
            "im selben Ordner liegen und dass das Notebook mit "
            "diesem Ordner als Arbeitsverzeichnis gestartet wurde."
        )

    print(f"{dataset_name.upper():8s} -> {file_path.name}")

print()
print(f"Ausgabeordner -> {OUTPUT_DIR}")

Arbeitsverzeichnis: /Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_additional_dataset/6_Datennormalisierung

CHN      -> chn_03_finale_titelannotationen.xlsx
MFC      -> mfc_03_finale_titelannotationen.xlsx
SEMEVAL  -> semeval_03_finale_titelannotationen_mit_3LLMs.xlsx

Ausgabeordner -> /Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_additional_dataset/6_Datennormalisierung/normalisierte_titelannotationen


# ## 2. Optionales Lowercasing
#
# Lowercasing ist aktuell absichtlich deaktiviert.
# Falls das konkrete ModernG-BERT-Setup später Lowercasing
# benötigt, kann die Funktion in `normalize_title()` aktiviert werden.


In [3]:
def lowercase_text(text: str) -> str:
    """
    Optionales Lowercasing.

    Aktuell NICHT aktiviert.
    """
    return text.lower()



# ## 3. Unsichtbare Unicode-Zeichen


In [4]:
def remove_invisible_unicode(text: str) -> str:
    """
    Entfernt unsichtbare Unicode-Formatzeichen.

    Sichtbare Zeichen wie ä, ö, ü, ß, €, –, —, -, „, “ usw.
    bleiben erhalten.
    """

    return "".join(
        char
        for char in text
        if unicodedata.category(char) != "Cf"
    )



# ## 4. Technische Normalisierung von `Headline_GER`


In [5]:
def normalize_title(value) -> str:
    """
    Technische Normalisierung von Headline_GER.

    Durchgeführt:
    1. Fehlende Werte -> leerer String
    2. Zeilenumbrüche und Tabs -> Leerzeichen
    3. Unsichtbare Unicode-Zeichen entfernen
    4. Unicode NFC-normalisieren
    5. Mehrfach-Whitespace -> einzelnes Leerzeichen
    6. Leerzeichen am Anfang/Ende entfernen

    Nicht durchgeführt:
    - Lowercasing
    - HTML-/Markup-Parsing
    - Übersetzung
    - Rechtschreibkorrektur
    - Entfernung sichtbarer Sonderzeichen
    - Entfernung von Umlauten
    - Entfernung von Interpunktion
    """

    if pd.isna(value):
        return ""

    text = str(value)

    # Zeilenumbrüche und Tabs normalisieren
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Unsichtbare Unicode-Zeichen entfernen
    text = remove_invisible_unicode(text)

    # Unicode konsistent normalisieren.
    text = unicodedata.normalize("NFC", text)

    # Mehrfach-Whitespace reduzieren
    text = re.sub(r"\s+", " ", text)

    # Leerzeichen am Anfang/Ende entfernen
    text = text.strip()

    return text


# ## 5. Einlesen und Strukturprüfung


In [6]:
def load_dataset(file_path: Path, sheet_name: str) -> pd.DataFrame:
    """Liest ein bestimmtes Excel-Sheet ein."""

    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        engine="openpyxl",
    )

    # Nur versehentliche Leerzeichen in Spaltennamen entfernen.
    df.columns = [str(col).strip() for col in df.columns]

    return df


def standardize_column_names(
    df: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Vereinheitlicht die Spaltennamen.

    Die aktuell verwendeten Dateien nutzen bereits
    `article_id` als gemeinsame ID-Spalte.
    """

    return df.copy()


def validate_dataset_structure(
    df: pd.DataFrame,
    dataset_name: str,
) -> None:
    """Prüft, ob alle erwarteten Spalten vorhanden sind."""

    required_columns = [
        ARTICLE_ID_COLUMN,
        TITLE_COLUMN,
        *LABEL_COLUMNS,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name}: Folgende erwartete Spalten fehlen:\n"
            + "\n".join(
                f"  - {column}"
                for column in missing_columns
            )
        )



6. Label-Signatur und Duplikatlogik

 Duplikate werden **nach der Titel-Normalisierung** geprüft.

 Regeln:

 1. Gleicher Titel + identischer vollständiger Label-Vektor:
    Eine Zeile bleibt erhalten.

 2. Gleicher Titel + unterschiedlicher Label-Vektor:
    Alle Zeilen dieses Titels werden entfernt.

 Die `article_id` ist NICHT Bestandteil der Duplikatdefinition.


In [7]:
def make_label_signature(row: pd.Series) -> tuple:
    """
    Erzeugt einen vollständigen Label-Vektor.
    """

    return tuple(
        row[column]
        for column in LABEL_COLUMNS
    )


def find_duplicate_groups(
    df: pd.DataFrame,
) -> tuple[set, set]:
    """
    Identifiziert Duplikatgruppen.

    Returns
    -------
    identical_duplicate_titles:
        Gleicher Titel + identische Annotation.

    conflicting_duplicate_titles:
        Gleicher Titel + unterschiedliche Annotation.
    """

    if df.empty:
        return set(), set()

    work = df[
        [TITLE_COLUMN] + LABEL_COLUMNS
    ].copy()

    work["_label_signature"] = work.apply(
        make_label_signature,
        axis=1,
    )

    identical_duplicate_titles = set()
    conflicting_duplicate_titles = set()

    for title, group in work.groupby(
        TITLE_COLUMN,
        sort=False,
        dropna=False,
    ):

        if len(group) <= 1:
            continue

        number_of_annotations = (
            group["_label_signature"]
            .nunique(dropna=False)
        )

        if number_of_annotations == 1:
            identical_duplicate_titles.add(title)
        else:
            conflicting_duplicate_titles.add(title)

    return (
        identical_duplicate_titles,
        conflicting_duplicate_titles,
    )


def remove_duplicates(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, dict]:
    """
    Wendet die vereinbarten Duplikatregeln an.

    Regel 1:
        Gleicher Titel + gleiche Annotation
        -> genau eine Zeile behalten.

    Regel 2:
        Gleicher Titel + unterschiedliche Annotation
        -> alle Zeilen dieses Titels entfernen.
    """

    if df.empty:
        return df.copy(), {
            "duplicate_title_groups_identical_annotation": 0,
            "duplicate_title_groups_conflicting_annotation": 0,
            "rows_removed_identical_annotation": 0,
            "rows_removed_conflicting_annotation": 0,
        }

    identical_titles, conflicting_titles = find_duplicate_groups(df)

    work = df.copy()

    # Konfligierende Titel vollständig entfernen.
    conflicting_mask = work[TITLE_COLUMN].isin(
        conflicting_titles
    )

    rows_removed_conflicting = int(
        conflicting_mask.sum()
    )

    work = work.loc[~conflicting_mask].copy()

    # Identische Duplikate:
    # pro Titel nur die erste Zeile behalten.
    identical_mask = work[TITLE_COLUMN].isin(
        identical_titles
    )

    identical_part = work.loc[
        identical_mask
    ].copy()

    non_duplicate_part = work.loc[
        ~identical_mask
    ].copy()

    rows_before_identical_removal = len(
        identical_part
    )

    identical_part = identical_part.drop_duplicates(
        subset=[
            TITLE_COLUMN,
            *LABEL_COLUMNS,
        ],
        keep="first",
    )

    rows_removed_identical = (
        rows_before_identical_removal
        - len(identical_part)
    )

    # Ursprüngliche Reihenfolge erhalten.
    cleaned = pd.concat(
        [
            non_duplicate_part,
            identical_part,
        ]
    ).sort_index()

    cleaned = cleaned.reset_index(drop=True)

    report = {
        "duplicate_title_groups_identical_annotation": len(
            identical_titles
        ),
        "duplicate_title_groups_conflicting_annotation": len(
            conflicting_titles
        ),
        "rows_removed_identical_annotation": (
            rows_removed_identical
        ),
        "rows_removed_conflicting_annotation": (
            rows_removed_conflicting
        ),
    }

    return cleaned, report



# ## 7. Qualitätskontrolle


In [8]:
def contains_invisible_unicode(text: str) -> bool:
    """
    Prüft, ob unsichtbare Unicode-Formatzeichen vorhanden sind.
    """

    if not isinstance(text, str):
        return False

    return any(
        unicodedata.category(char) == "Cf"
        for char in text
    )


def create_quality_report(
    original_df: pd.DataFrame,
    normalized_df: pd.DataFrame,
    final_df: pd.DataFrame,
    duplicate_report: dict,
    dataset_name: str,
) -> dict:

    original_titles = (
        original_df[TITLE_COLUMN]
        .fillna("")
        .astype(str)
    )

    normalized_titles = (
        normalized_df[TITLE_COLUMN]
        .fillna("")
        .astype(str)
    )

    changed_titles = (
        original_titles.reset_index(drop=True)
        != normalized_titles.reset_index(drop=True)
    )

    # Technische Auffälligkeiten im Original
    invisible_unicode_count = sum(
        contains_invisible_unicode(title)
        for title in original_titles
    )

    newline_count = sum(
        bool(re.search(r"[\r\n]", title))
        for title in original_titles
    )

    tab_count = sum(
        "\t" in title
        for title in original_titles
    )

    final_titles = (
        final_df[TITLE_COLUMN]
        .fillna("")
        .astype(str)
    )

    title_lengths = final_titles.str.len()

    final_invisible_unicode_count = sum(
        contains_invisible_unicode(title)
        for title in final_titles
    )

    final_newline_count = sum(
        bool(re.search(r"[\r\n]", title))
        for title in final_titles
    )

    final_tab_count = sum(
        "\t" in title
        for title in final_titles
    )

    return {
        "Dataset": dataset_name,

        "Zeilen_original": len(original_df),

        "Zeilen_nach_Normalisierung": len(
            normalized_df
        ),

        "Zeilen_final": len(final_df),

        "Entfernte_Zeilen_gesamt": (
            len(normalized_df) - len(final_df)
        ),

        "Leere_Titel_original": int(
            (original_titles.str.strip() == "").sum()
        ),

        "Leere_Titel_final": int(
            (final_titles.str.strip() == "").sum()
        ),

        "Titel_durch_Normalisierung_verändert": int(
            changed_titles.sum()
        ),

        "Titel_mit_unsichtbaren_Unicodezeichen": (
            invisible_unicode_count
        ),

        "Titel_mit_Zeilenumbrüchen": newline_count,

        "Titel_mit_Tabs": tab_count,

        "Verbleibende_unsichtbare_Unicodezeichen_final": (
            final_invisible_unicode_count
        ),

        "Verbleibende_Zeilenumbrüche_final": (
            final_newline_count
        ),

        "Verbleibende_Tabs_final": (
            final_tab_count
        ),

        "Min_Titellänge": (
            int(title_lengths.min())
            if len(title_lengths)
            else 0
        ),

        "Max_Titellänge": (
            int(title_lengths.max())
            if len(title_lengths)
            else 0
        ),

        "Durchschnittliche_Titellänge": round(
            float(title_lengths.mean()), 2
        ) if len(title_lengths) else 0,

        **duplicate_report,
    }



# ## 8. Datensätze verarbeiten


In [9]:
processed_datasets = {}
quality_reports = {}

for dataset_name, config in DATASET_CONFIG.items():

    print("=" * 70)
    print(f"Verarbeite: {dataset_name.upper()}")
    print("=" * 70)

    # Excel einlesen
    original_df = load_dataset(
        INPUT_FILES[dataset_name],
        config["sheet"],
    )

    # Spaltennamen vereinheitlichen
    original_df = standardize_column_names(
        original_df,
        dataset_name,
    )

    # Struktur prüfen
    validate_dataset_structure(
        original_df,
        dataset_name,
    )

    print(
        f"Originale Zeilen: {len(original_df):,}"
    )

    # Titel technisch normalisieren
    normalized_df = original_df.copy()

    normalized_df[TITLE_COLUMN] = (
        normalized_df[TITLE_COLUMN]
        .apply(normalize_title)
    )

    # Duplikate NACH der Normalisierung entfernen
    cleaned_df, duplicate_report = remove_duplicates(
        normalized_df
    )

    # Qualitätsbericht erzeugen
    report = create_quality_report(
        original_df=original_df,
        normalized_df=normalized_df,
        final_df=cleaned_df,
        duplicate_report=duplicate_report,
        dataset_name=dataset_name,
    )

    processed_datasets[dataset_name] = cleaned_df
    quality_reports[dataset_name] = report

    print(
        f"Finale Zeilen:   {len(cleaned_df):,}"
    )

    print(
        f"Entfernte Zeilen: "
        f"{len(normalized_df) - len(cleaned_df):,}"
    )

    print(
        "  davon identische Annotation: "
        f"{duplicate_report['rows_removed_identical_annotation']:,}"
    )

    print(
        "  davon widersprüchliche Annotation: "
        f"{duplicate_report['rows_removed_conflicting_annotation']:,}"
    )

    print()



Verarbeite: CHN
Originale Zeilen: 312
Finale Zeilen:   311
Entfernte Zeilen: 1
  davon identische Annotation: 1
  davon widersprüchliche Annotation: 0

Verarbeite: MFC
Originale Zeilen: 6,214
Finale Zeilen:   6,082
Entfernte Zeilen: 132
  davon identische Annotation: 132
  davon widersprüchliche Annotation: 0

Verarbeite: SEMEVAL


ValueError: Worksheet named 'Titelannotation_wide' not found

# ## 9. Drei normalisierte Excel-Dateien speichern


In [ ]:
for dataset_name, df in processed_datasets.items():

    output_path = OUTPUT_FILES[dataset_name]

    df.to_excel(
        output_path,
        index=False,
        engine="openpyxl",
    )

    print(
        f"{dataset_name.upper():8s} -> {output_path}"
    )



# ## 10. Kombiniertes Dataset erstellen
#
# SemEval wird aus dem Worksheet `Sheet1` übernommen.
#
# Die gemeinsame Struktur verwendet `article_id` und `Headline_GER`.
# Eine neue `row_id` wird ganz vorne eingefügt und von 1 an
# fortlaufend nummeriert.


In [ ]:
# Alle Spalten sammeln, die in mindestens einem Datensatz
# vorhanden sind.
all_columns = []

for df in processed_datasets.values():

    for column in df.columns:

        if column not in all_columns:
            all_columns.append(column)


# article_id soll an erster Stelle der eigentlichen Daten stehen.
if ARTICLE_ID_COLUMN in all_columns:
    all_columns.remove(ARTICLE_ID_COLUMN)
    all_columns.insert(0, ARTICLE_ID_COLUMN)


# Headline_GER direkt hinter article_id.
if TITLE_COLUMN in all_columns:
    all_columns.remove(TITLE_COLUMN)

    insert_position = (
        1
        if ARTICLE_ID_COLUMN in all_columns
        else 0
    )

    all_columns.insert(
        insert_position,
        TITLE_COLUMN,
    )


# Datensätze auf gemeinsame Struktur bringen
aligned_datasets = []

for dataset_name, df in processed_datasets.items():

    aligned_df = df.reindex(
        columns=all_columns
    )

    aligned_datasets.append(aligned_df)


# Zusammenführen
combined_df = pd.concat(
    aligned_datasets,
    ignore_index=True,
)


# Neue fortlaufende row_id ganz vorne
combined_df.insert(
    0,
    "row_id",
    range(1, len(combined_df) + 1),
)


print(
    f"Kombinierter Datensatz: "
    f"{len(combined_df):,} Zeilen × "
    f"{len(combined_df.columns):,} Spalten"
)



# ## 11. Kombinierte Datei speichern


In [ ]:
combined_df.to_excel(
    OUTPUT_FILES["combined"],
    index=False,
    engine="openpyxl",
)

print(
    f"Kombinierte Datei gespeichert:\n"
    f"{OUTPUT_FILES['combined']}"
)



# ## 12. Qualitätsbericht anzeigen


In [ ]:
quality_report_df = pd.DataFrame(
    quality_reports.values()
)

preferred_columns = [
    "Dataset",
    "Zeilen_original",
    "Zeilen_nach_Normalisierung",
    "Zeilen_final",
    "Entfernte_Zeilen_gesamt",
    "Titel_durch_Normalisierung_verändert",
    "Titel_mit_unsichtbaren_Unicodezeichen",
    "Titel_mit_Zeilenumbrüchen",
    "Titel_mit_Tabs",
    "Verbleibende_unsichtbare_Unicodezeichen_final",
    "Verbleibende_Zeilenumbrüche_final",
    "Verbleibende_Tabs_final",
    "Leere_Titel_original",
    "Leere_Titel_final",
    "Min_Titellänge",
    "Max_Titellänge",
    "Durchschnittliche_Titellänge",
    "duplicate_title_groups_identical_annotation",
    "rows_removed_identical_annotation",
    "duplicate_title_groups_conflicting_annotation",
    "rows_removed_conflicting_annotation",
]

quality_report_df = quality_report_df[
    [
        column
        for column in preferred_columns
        if column in quality_report_df.columns
    ]
]

display(quality_report_df)



# ## 13. Abschlusskontrolle


In [ ]:
print("=" * 70)
print("ABSCHLUSSKONTROLLE")
print("=" * 70)

print("\nAusgabedateien:")

for name, path in OUTPUT_FILES.items():

    exists = path.exists()

    print(
        f"  {name:10s}: "
        f"{'OK' if exists else 'FEHLT'} "
        f"-> {path}"
    )


print("\nKombinierter Datensatz:")

print(
    f"  Zeilen: {len(combined_df):,}"
)

print(
    f"  Spalten: {len(combined_df.columns):,}"
)

print(
    f"  row_id vorhanden: "
    f"{'row_id' in combined_df.columns}"
)

print(
    f"  Headline_GER vorhanden: "
    f"{TITLE_COLUMN in combined_df.columns}"
)

print("\nVerarbeitung abgeschlossen.")
